In [1]:
import requests
import time
import base64
from PIL import Image

In [3]:
#sanity check with /v1/models
# base url example: base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"
# enter the inference endpoints/url from the deployed model

base_url = "<enter your base url here>"
requests.get(f"{base_url}/v1/models").json()

{'object': 'list',
 'data': [{'id': 'granite-vision-model',
   'object': 'model',
   'created': 1785882784,
   'owned_by': 'vllm',
   'root': '/mnt/models',
   'parent': None,
   'max_model_len': 8192,
   'permission': [{'id': 'modelperm-ad4677672ed33d5a',
     'object': 'model_permission',
     'created': 1785882784,
     'allow_create_engine': False,
     'allow_sampling': True,
     'allow_logprobs': True,
     'allow_search_indices': False,
     'allow_view': True,
     'allow_fine_tuning': False,
     'organization': '*',
     'group': None,
     'is_blocking': False}]}]}

In [4]:
#Perform a basic text-only test

base_url = "http://granite-vision-model-predictor.user9.svc.cluster.local"

resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [{"role": "user", "content": "Hello, are you working?"}],
        "max_tokens": 100
    }
)
print(resp.status_code)
print(resp.json())

200
{'id': 'chatcmpl-92218f3dccefd52a', 'object': 'chat.completion', 'created': 1785882787, 'model': 'granite-vision-model', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Greetings! I'm here and ready to assist you. How can I help you today?", 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 54, 'total_tokens': 75, 'completion_tokens': 21, 'prompt_tokens_details': None}, 'prompt_logprobs': None, 'prompt_token_ids': None, 'kv_transfer_params': None}


In [5]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_b64 = encode_image("chart.png")
print("Encoded length:", len(image_b64))  # sanity check — should be a long string, not empty

Encoded length: 74224


In [7]:
#Perform a multimodal test - send an image + question

# Check image size first
img = Image.open("chart.png")
print("Image size:", img.size)

#For a 619x344 image on CPU-only inference the range should 30-90 seconds for inference response.
start = time.time()
resp = requests.post(
    f"{base_url}/v1/chat/completions",
    json={
        "model": "granite-vision-model",
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": "What does this chart show?"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}
        ],
        "max_tokens": 100
    },
    timeout=120
)
print("Elapsed:", time.time() - start, "seconds")
print(resp.status_code)
print(resp.json())

Image size: (619, 344)
Elapsed: 61.57270336151123 seconds
200
{'id': 'chatcmpl-9ffbfc327459aa49', 'object': 'chat.completion', 'created': 1785882867, 'model': 'granite-vision-model', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'This is a diagram showing only key elements for both data centers. At the heart of this architecture is the shared underlying storage system for the two clusters. This is the only shared resource on the system and is accessed through the shared SAN. After the storage is provisioned, database instances are created as normal.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'stop', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 2107, 'total_tokens': 2170, 'completion_tokens': 63, 'prompt_tokens_details': None}, 'prompt_logprobs': None, 'prompt_token_ids': None, 'kv_tra


<b>Let's talk Response Time </b>
The response quality (60-90s) is genuinely good — it correctly identified the image as an architecture diagram, described the high-availability/replication concept, and mentioned load balancers and data center replication — that's real diagram comprehension, not a generic answer.

In [8]:
#test the embedding model

# Perform a basic embedding test
embed_base_url = "http://redhataiall-minilm-l6-v2-predictor.user9.svc.cluster.local"

resp = requests.post(
    f"{embed_base_url}/v1/embeddings",
    json={
        "model": "redhataiall-minilm-l6-v2",
        "input": "Hello, are you working?"
    }
)

print(resp.status_code)

data = resp.json()
embedding = data["data"][0]["embedding"]
print("Embedding length:", len(embedding))       # should print 384
print("First 5 values:", embedding[:5])

200
Embedding length: 384
First 5 values: [-0.0619322806596756, 0.00833729188889265, 0.05295676738023758, 0.03518441691994667, -0.03582704812288284]


In [9]:
import numpy as np

def get_embedding(text):
    resp = requests.post(
        f"{embed_base_url}/v1/embeddings",
        json={"model": "redhataiall-minilm-l6-v2", "input": text}
    )
    return np.array(resp.json()["data"][0]["embedding"])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

e1 = get_embedding("The cat sat on the mat.")
e2 = get_embedding("A feline rested on the rug.")     # similar meaning
e3 = get_embedding("The stock market crashed today.")  # unrelated meaning

print("Similar sentences:", cosine_similarity(e1, e2))    # should be high, e.g. 0.6-0.9
print("Unrelated sentences:", cosine_similarity(e1, e3))  # should be much lower

Similar sentences: 0.5561058456567265
Unrelated sentences: 0.09435894834611532


<b>Test semantic similarity to prove the embedding  model is working properly.</b>

Confirms the model actually understands meaning, not just that the API responds.

In our example, the "similar sentences" score comes back noticeably higher than the "unrelated sentences" score, that's solid confirmation the embedding model is working correctly end-to-end.